In [1]:
import gzip

rsid_file = "/scratch1/jazlynmo/cancerproj/processGWAScatalog/processed_gwas-associations_SNPIDCurrent_rsIDs.txt"
snp_file  = "/scratch1/jazlynmo/cancerproj/snp151.txt.gz"
out_file  = "rsid_positions_hg38.txt"

# ---- Load rsIDs into a set ----
with open(rsid_file) as f:
    rsids = set(line.strip() for line in f if line.strip())

original_count = len(rsids)
print(f"Loaded {original_count} rsIDs")


Loaded 8207 rsIDs


In [2]:
# ---- Stream through gzip and filter ----
with gzip.open(snp_file, "rt") as infile, open(out_file, "w") as out:

    write = out.write
    write("rsID\tchr\tstart\tend\talleles\n")

    for line in infile:
        cols = line.split("\t")

        if len(cols) < 10:
            continue

        rsid = cols[4]

        if rsid in rsids:
            write(f"{rsid}\t{cols[1]}\t{cols[2]}\t{cols[3]}\t{cols[9]}\n")

            rsids.remove(rsid)

            if not rsids:
                print("All rsIDs found — stopping early")
                break

# ---- Report missing ----
missing = len(rsids)
found = original_count - missing

print(f"Found {found} rsIDs")
print(f"Missing {missing} rsIDs")

# Optional: write missing list for inspection
if missing:
    with open("rsids_not_found.txt", "w") as mf:
        for r in sorted(rsids):
            mf.write(r + "\n")

    print("Missing rsIDs written to rsids_not_found.txt")

print("Done.")

Found 8195 rsIDs
Missing 12 rsIDs
Missing rsIDs written to rsids_not_found.txt
Done.
